# LifeMate — nhánh chính Zipformer Việt 68M / Sherpa

Input dataset cũ, GPU + Internet → Run All. Không cần HF_TOKEN. Full fine-tuning có giám sát;
Sherpa là runtime triển khai, model train là zzasdf/viet_iter3_pseudo_label.
Giữ 5036 audio: 4532 train / 504 valid; không gọi valid là test độc lập hoặc speaker-disjoint.
Đây là một nhánh chính có baseline và checkpoint riêng, không phải phương án dự phòng.

Checkpoint gốc epoch-12-avg-8.pt, revision e827965a37aab92a4455566fac49c0e80a23afef.
Đã kiểm tra tokenizer gốc: 442 train / 39 valid có UNK vì 0–9. Bản này thêm cố định
10 token số, không fit tokenizer bằng valid; copy nguyên các trọng số và ID token cũ.
Token số mới khởi tạo từ token từ đọc số tương ứng, joiner bias giảm 2 để tránh lấn át từ cũ.
Baseline che logits token mới; fine-tune cho phép học số. Reference/chuẩn hóa chấm điểm giữ nguyên.
Phải dùng bpe.model được xuất kèm checkpoint mới, không dùng lại BPE gốc cho model 2010 token.

Train RNN-T: 0.5 simple + pruned loss; không thêm head CTC/CR ngẫu nhiên.
Speed 0.9/1.0/1.1, SpecAugment nhẹ, nhiễu Gaussian 20% (SNR 15–25 dB),
ưu tiên câu chứa từ tiếng Anh theo lexicon cố định và nối nguyên câu có nhãn đồng bộ.
Gaussian không thay thế noise thực. Những augmentation này không giả lập giọng miền Trung/nói ngọng.
Epoch 1 học đủ train; mining lỗi thiếu từ trên train rồi thêm tối đa 20% lượt replay từ epoch 2.
Đây là biến thể lấy cảm hứng JTT, không tái hiện JTT và không tự suy ra nhãn giọng.
EM cao nhất, hòa lấy WER thấp hơn; patience 2; lưu last.pt mỗi 25 updates và best.pt theo valid.
Ngân sách 9 giờ bắt đầu sau baseline, bao gồm valid/mining giữa các epoch; kiểm tra sau optimizer update.
Validation khi dừng có thể làm vượt mốc này. Không hứa giờ hoàn thành hay mức tăng.

Cơ sở nghiên cứu (rank hội nghị theo CORE2023, không phải rank từng bài):
- [Zipformer, ICLR 2024 / A*](https://openreview.net/pdf?id=9WD9KwssyT): kiến trúc và ScaledAdam.
- [JTT, ICML 2021 / A*](https://proceedings.mlr.press/v139/liu21f.html): ý tưởng tìm lỗi train rồi ưu tiên học lại;
  paper gốc không thử ASR. Bản này tiếp tục weights thay vì train lại từ đầu như JTT.
- [ViMD, EMNLP 2024 / A*](https://aclanthology.org/2024.emnlp-main.426/): đánh giá đa giọng cần dữ liệu/nhãn thật;
  chưa nhập ViMD hoặc có nhãn giọng để báo cải thiện miền Trung.
- [SpecAugment, Interspeech 2019 / A](https://www.isca-archive.org/interspeech_2019/park19e_interspeech.html),
  [speed perturbation, Interspeech 2015 / A](https://www.isca-archive.org/interspeech_2015/ko15_interspeech.html): nguồn bổ trợ,
  không ghi thành A*/Q1/Q2.
Nguồn rank: [ICLR](https://portal.core.edu.au/conf-ranks/2273/),
[ICML](https://portal.core.edu.au/conf-ranks/1121/), [EMNLP](https://portal.core.edu.au/conf-ranks/448/).
Mở rộng token số, LR 0.002, tỷ lệ replay và augmentation là lựa chọn kỹ thuật cần kiểm nghiệm LifeMate.
Chưa có kết quả train GPU cho nhánh này; chỉ báo kết quả sau preflight/validation thực tế.


## Cell 1 — Cấu hình nhánh chính 68M: 5 epoch, EM trước WER, patience 2, chạy mới.


In [ ]:
from pathlib import Path
DATASET_ROOT = Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
WORK_DIR = Path('/kaggle/temp/lifemate_sherpa_vi_finetune')
OUTPUT_DIR = Path('/kaggle/working/lifemate_sherpa_vi_68m_main_5epoch_em')
SELECTION_METRIC = 'normalized_exact_match_then_wer'
RESUME_DIR = None  # Path('/kaggle/input/.../lifemate_hynt_finetune')
AUTO_RESUME_DISCOVERY = False
ALLOW_CODE_FIX_RESUME = False  # Chỉ nhận checkpoint có cùng split, architecture và core hyperparameters.
SEED = 2026
VALID_FRACTION = 0.10
EPOCHS = 5
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 0.002  # ScaledAdam; 1/10 LR 0.04 trong checkpoint tác giả.
PATIENCE = 2
MAX_HOURS = 9.0  # Tạm dừng và lưu last.pt khi hết khoảng train này.
SAVE_EVERY_UPDATES = 25
ENGLISH_EXTRA_FRACTION = 0.5
CS_CONCAT_PROBABILITY = 0.10
NOISE_PROBABILITY = 0.20
ICEFALL_COMMIT = '9c966762e058bd7ba79d9da71e42fc06b64b703e'
K2_WHEEL_URL = 'https://huggingface.co/csukuangfj/k2/resolve/f9222235425ce5eee567edda570c0abac4607ff0/ubuntu-cuda/k2-1.24.4.dev20250715+cuda12.4.torch2.5.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl#sha256=79e62f29bd0964f3919fe224385fb2e29e3d3513a4085b65bdd259475411722b'
HARD_EXTRA_FRACTION = 0.20  # Extra train-only replay after epoch 1
AUGMENT_TRAIN = True  # False for the plain fine-tuning ablation


## Cell 2 — GPU và Python 3.11 riêng. Đọc dòng RUN nếu setup bị kẹt.


In [ ]:
import sys, subprocess, os, json, shutil, time
WORK_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
ENV = dict(os.environ, PYTHONUNBUFFERED='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1',
           HF_HUB_DISABLE_XET='1', HF_XET_CHUNK_CACHE_SIZE_BYTES='0',
           HF_HOME=str(WORK_DIR/'hf'), UV_CACHE_DIR=str(WORK_DIR/'uv_cache'),
           UV_PYTHON_INSTALL_DIR=str(WORK_DIR/'managed_python'), OMP_NUM_THREADS='2')
# Kaggle may export UV_SYSTEM_PYTHON; explicit --python below owns this env.
ENV.pop('UV_SYSTEM_PYTHON',None)
def run(command, cwd=None):
    print('RUN:', ' '.join(str(x) for x in command),flush=True)
    with (OUTPUT_DIR/'run.log').open('a',encoding='utf-8') as log:
        process=subprocess.Popen([str(x) for x in command],cwd=cwd,env=ENV,
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        if process.wait()!=0:raise RuntimeError('Process failed; see run.log / failure.json in Output')
run(['nvidia-smi'])
run([sys.executable,'-m','pip','install','uv'])
VENV=WORK_DIR/'venv'
PYTHON=VENV/'bin/python'
if not PYTHON.exists():run([sys.executable,'-m','uv','venv','--python','3.11','--managed-python',VENV])

## Cell 3 — Torch 2.5.1/k2 CUDA, SentencePiece và protobuf.


In [ ]:
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,
     'torch==2.5.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu124'])
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,
     'protobuf==5.29.5','numpy==1.26.4','lhotse==1.32.0','soundfile==0.13.1','sentencepiece==0.2.1',
     'huggingface-hub==1.24.0','rapidfuzz==3.14.1','kaldialign==0.9.1',
     'pypinyin==0.50.0','tensorboard==2.19.0'])
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'--no-deps',K2_WHEEL_URL])
run([PYTHON,'-c',
     'import torch,torchaudio,k2,lhotse; from importlib.metadata import version; '
     'assert torch.cuda.is_available(), "Enable Kaggle GPU"; '
     'assert torch.__version__.startswith("2.5.1") and torch.version.cuda=="12.4"; '
     'assert k2.with_cuda, "k2 CPU wheel is not sufficient"; '
     'print("CUDA imports OK:",torch.cuda.get_device_name(0),torch.__version__,version("k2"))'])

## Cell 4 — Khóa Icefall và kiểm tra import.


In [ ]:
ICEFALL_ROOT=WORK_DIR/'icefall'
if not (ICEFALL_ROOT/'.git').exists():
    ICEFALL_ROOT.mkdir(exist_ok=True)
    run(['git','init',ICEFALL_ROOT])
    run(['git','remote','add','origin','https://github.com/k2-fsa/icefall.git'],cwd=ICEFALL_ROOT)
head=subprocess.run(['git','rev-parse','HEAD'],cwd=ICEFALL_ROOT,text=True,capture_output=True).stdout.strip()
if head!=ICEFALL_COMMIT:
    run(['git','fetch','--depth','1','origin',ICEFALL_COMMIT],cwd=ICEFALL_ROOT)
    run(['git','checkout','--detach','FETCH_HEAD'],cwd=ICEFALL_ROOT)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ICEFALL_ROOT,text=True).strip()==ICEFALL_COMMIT
run([PYTHON,'-c','import sys;sys.path[:0]=[sys.argv[1],sys.argv[2]];from model import AsrModel;from optim import ScaledAdam;print("ICEFALL_IMPORT_OK")',ICEFALL_ROOT/'egs/librispeech/ASR/zipformer',ICEFALL_ROOT])

## Cell 5 — Trainer RNN-T, mở rộng 10 ký tự số và giữ toàn bộ trọng số cũ.


In [ ]:
SOURCES = {
    'sherpa_vi_finetune.py': r'''"""Principal LifeMate Vietnamese Zipformer 68M supervised RNN-T fine-tuning.

Digit vocabulary adaptation and train-only hard-example replay are explicit
engineering adaptations, not claimed reproductions of JTT or accent labels.
"""
import argparse
import csv
import gc
import hashlib
import json
import math
import logging
from pathlib import Path, PosixPath
import random
import sys
import time

from hynt_finetune_data import normalize, save_json, sha_file, stable_hash

MODEL_ID = 'zzasdf/viet_iter3_pseudo_label'
REVISION = 'e827965a37aab92a4455566fac49c0e80a23afef'
CHECKPOINT_SHA = 'b81cf36432463fed2b3615dfec955fa54f64eed45981729c35d8ccde98ddeef2'
BPE_SHA = '289dbb44527c13c419ae3a4d8ce6a349f01a97f8777e69934a77e3692d2f10db'
SELECTION_METRIC = 'normalized_exact_match_then_wer'


def better_validation_score(em,wer,best_em,best_wer):
    if not math.isfinite(em) or not 0<=em<=1 or not math.isfinite(wer) or wer<0:
        raise ValueError('Invalid validation score')
    return best_em is None or em>best_em or (em==best_em and wer<best_wer)


def require_em_checkpoint(saved):
    if saved.get('selection_metric')!=SELECTION_METRIC or saved.get('best_em') is None:
        raise ValueError('Old WER-selected run cannot resume as EM-selected: historical best-EM weights are unknown. Use a new OUTPUT_DIR for a fresh run; keep the old checkpoints.')


class NonfiniteTrainingLoss(FloatingPointError):
    def __init__(self,details):
        self.details=details
        super().__init__('Nonfinite training loss: '+json.dumps(details,ensure_ascii=False))


def recover_loss(attempts,on_failure):
    """Retry the same row; never replace a failed loss by zero or advance it."""
    failures=[]
    for name,calculate in attempts:
        try:return calculate(),name
        except NonfiniteTrainingLoss as error:
            failures.append(dict(strategy=name,**error.details))
            on_failure(failures[-1])
    raise NonfiniteTrainingLoss(dict(all_attempts_failed=failures))


def sherpa_params():
    # Weight-dependent dimensions were checked against every publisher tensor.
    # Unmasked dimensions and schedule are our fine-tuning choices, not metadata
    # supplied by the author (the published checkpoint contains only model).
    return dict(feature_dim=80,num_encoder_layers='2,2,3,4,3,2',
        downsampling_factor='1,2,4,8,4,2',feedforward_dim='512,768,1024,1536,1024,768',
        num_heads='4,4,4,8,4,4',encoder_dim='192,256,384,512,384,256',
        query_head_dim='32',value_head_dim='12',pos_head_dim='4',pos_dim=48,
        encoder_unmasked_dim='192,192,256,256,256,192',cnn_module_kernel='31,31,15,15,15,31',
        decoder_dim=512,joiner_dim=512,causal=False,chunk_size='-1',left_context_frames='-1',
        context_size=2,blank_id=0,vocab_size=2000)


def extend_digit_bpe(source,target):
    from sentencepiece import sentencepiece_model_pb2 as pb
    model=pb.ModelProto();model.ParseFromString(Path(source).read_bytes())
    existing={p.piece for p in model.pieces};added=[]
    for char in '0123456789':  # Fixed inventory; never learn a vocabulary from valid.
        if char not in existing:
            piece=model.pieces.add();piece.piece=char;piece.score=0.;piece.type=4
            added.append(char)
    model.trainer_spec.vocab_size=len(model.pieces)
    Path(target).write_bytes(model.SerializeToString())
    return added


def expand_digit_weights(torch,original,initialized,old_vocab=2000,digit_source_ids=None):
    expandable={'decoder.embedding.weight','joiner.output_linear.weight','joiner.output_linear.bias',
                'simple_am_proj.weight','simple_am_proj.bias','simple_lm_proj.weight','simple_lm_proj.bias'}
    result={}
    if set(original)!=set(initialized):raise ValueError('Unexpected model state keys')
    for name,dst in initialized.items():
        src=original[name]
        if src.shape==dst.shape:result[name]=src;continue
        if (name not in expandable or src.shape[0]!=old_vocab or dst.shape[0]!=old_vocab+10
                or src.shape[1:]!=dst.shape[1:]):
            raise ValueError('Unexpected vocabulary resize: '+name)
        dst=dst.clone();dst[:old_vocab].copy_(src)
        if digit_source_ids is not None:
            if len(digit_source_ids)!=10 or any(not 0<=i<old_vocab for i in digit_source_ids):
                raise ValueError('Invalid digit initialization IDs')
            dst[old_vocab:].copy_(src[digit_source_ids])
            if name=='joiner.output_linear.bias':dst[old_vocab:]-=2.
        elif dst.ndim==1:dst[old_vocab:].fill_(-10. if name=='joiner.output_linear.bias' else 0.)
        else:dst[old_vocab:].normal_(mean=0.,std=.01)
        result[name]=dst
    return result


def replay_order(rows,seed,epoch,hard_uids,extra_english=.5,extra_hard=.2):
    order=epoch_order(rows,seed,epoch,extra_english)
    hard_set=set(hard_uids)
    eligible=[i for i,row in enumerate(rows) if row['uid'] in hard_set]
    if set(hard_uids)-{row['uid'] for row in rows}:raise ValueError('Hard pool contains non-train IDs')
    # Sampling without replacement caps each hard example at one extra replay.
    rng=random.Random(seed+epoch*131071)
    order+=rng.sample(eligible,min(len(eligible),round(len(rows)*extra_hard)))
    rng.shuffle(order)
    return order


def compatible_resume_metadata(resume,output,saved_identity,cfg,split_fingerprint):
    """Explicit code-fix migration, retaining the source identity and protocol."""
    core=('icefall_commit','seed','epochs','gradient_accumulation','learning_rate','patience',
          'english_extra_fraction','cs_concat_probability','noise_probability')
    if not isinstance(saved_identity,str) or len(saved_identity)!=64 or any(c not in '0123456789abcdef' for c in saved_identity):
        raise ValueError('Invalid previous checkpoint identity')
    for folder in dict.fromkeys([Path(resume).parent,Path(output)]):
        for name in ['run_identity.json',f'previous_run_{saved_identity}.json']:
            path=folder/name
            if not path.is_file():continue
            prior=json.loads(path.read_text(encoding='utf-8'))
            if (prior.get('identity')==saved_identity and prior.get('split_fingerprint')==split_fingerprint
                    and all(prior.get('config',{}).get(k)==cfg.get(k) for k in core)):
                return prior
    raise ValueError('Previous-code checkpoint has no matching split/hyperparameter provenance; refusing migration')


def construct_model(params):
    import torch
    from decoder import Decoder
    from joiner import Joiner
    from model import AsrModel
    from scaling import ScheduledFloat
    from subsampling import Conv2dSubsampling
    from zipformer import Zipformer2
    ints=lambda key:tuple(int(x) for x in str(params[key]).split(','))
    dims=ints('encoder_dim')
    embed=Conv2dSubsampling(in_channels=params['feature_dim'],out_channels=dims[0],
                           dropout=ScheduledFloat((0.,.3),(20000.,.1)))
    names=('downsampling_factor','num_encoder_layers','encoder_dim','encoder_unmasked_dim',
           'query_head_dim','pos_head_dim','value_head_dim','num_heads','feedforward_dim',
           'cnn_module_kernel','chunk_size','left_context_frames')
    encoder=Zipformer2(output_downsampling_factor=2,**{k:ints(k) for k in names},
        pos_dim=params['pos_dim'],dropout=ScheduledFloat((0.,.3),(20000.,.1)),
        warmup_batches=4000.,causal=params['causal'])
    decoder=Decoder(vocab_size=params['vocab_size'],decoder_dim=params['decoder_dim'],
                    blank_id=params['blank_id'],context_size=params['context_size'])
    joiner=Joiner(encoder_dim=max(dims),decoder_dim=params['decoder_dim'],
                  joiner_dim=params['joiner_dim'],vocab_size=params['vocab_size'])
    return AsrModel(encoder_embed=embed,encoder=encoder,decoder=decoder,joiner=joiner,
        encoder_dim=max(dims),decoder_dim=params['decoder_dim'],vocab_size=params['vocab_size'],
        use_transducer=True,use_ctc=False,use_attention_decoder=False)



def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


def atomic_torch_save(torch, data, path):
    path=Path(path);temp=path.with_suffix('.tmp')
    torch.save(data,temp);temp.replace(path)


def check_finite_weights(torch,model):
    checks=[torch.isfinite(p).all() for p in model.parameters()]
    if checks and not torch.stack(checks).all().item():
        raise FloatingPointError('Nonfinite model weights; previous last.pt/best.pt are kept untouched')


def training_snapshot(torch,model,optimizer,scaler,identity,params,**progress):
    return dict(identity=identity,model=model.state_dict(),optimizer=optimizer.state_dict(),
        scaler=scaler.state_dict(),params=params,torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),python_rng=random.getstate(),**progress)


def restore_training_snapshot(torch,saved,model,optimizer,scaler,identity):
    if saved['identity']!=identity:raise ValueError('Resume checkpoint belongs to another config/split/source')
    model.load_state_dict(saved['model'],strict=True)
    optimizer.load_state_dict(saved['optimizer']);scaler.load_state_dict(saved['scaler'])
    torch.set_rng_state(saved['torch_rng'])
    # Training uses cuda:0 only; an unused GPU's saved RNG must not prevent
    # resuming a T4x2 session on a single-GPU session.
    torch.cuda.set_rng_state_all(saved['cuda_rng'][:torch.cuda.device_count()])
    random.setstate(saved['python_rng'])
    result={k:saved[k] for k in ('global_step','epoch','cursor','best_wer','bad_epochs')}
    if 'best_em' in saved:result['best_em']=saved['best_em']
    return result


def set_batch_count(model, count):
    for name,module in model.named_modules():
        if hasattr(module,'batch_count'):module.batch_count=float(count)
        if hasattr(module,'name'):module.name=name


def epoch_order(rows, seed, epoch, extra_english=.5):
    rng=random.Random(seed+epoch*100003)
    order=list(range(len(rows)))
    english=[i for i,r in enumerate(rows) if r['english_term_hint']]
    if english: order+=rng.choices(english,k=round(len(english)*extra_english))
    rng.shuffle(order)
    return order


def main(config_path):
    import importlib.metadata
    import numpy as np
    import soundfile as sf
    import torch
    import torchaudio
    import k2
    from huggingface_hub import hf_hub_download
    from lhotse import Fbank,FbankConfig
    from lhotse.dataset import SpecAugment
    from rapidfuzz.distance import Levenshtein
    import sentencepiece as spm

    cfg=json.loads(Path(config_path).read_text(encoding='utf-8'))
    out,work=Path(cfg['output']),Path(cfg['work'])
    out.mkdir(parents=True,exist_ok=True);work.mkdir(parents=True,exist_ok=True)
    logging.basicConfig(level=logging.WARNING)
    detail_log=logging.FileHandler(out/'optimizer_warnings.log',encoding='utf-8')
    logging.getLogger().addHandler(detail_log)
    class OptimizerConsoleFilter(logging.Filter):
        def filter(self,record):
            return not record.getMessage().startswith(('Clipping_scale=','Scaling gradients by ',
                'Parameter dominating tot_sumsq ','Parameters with most larger-than-usual grads'))
    for handler in logging.getLogger().handlers:
        if isinstance(handler,logging.StreamHandler) and not isinstance(handler,logging.FileHandler):
            handler.addFilter(OptimizerConsoleFilter())
    if not torch.cuda.is_available():raise RuntimeError('Enable Kaggle GPU T4/P100; CUDA is required')
    if '2.5.1' not in torch.__version__ or torch.version.cuda!='12.4':
        raise RuntimeError('Use isolated notebook runtime torch 2.5.1+cu124')
    torch.set_num_threads(2)
    torch.manual_seed(cfg['seed']);torch.cuda.manual_seed_all(cfg['seed'])
    random.seed(cfg['seed']);np.random.seed(cfg['seed'])
    recipe=Path(cfg['icefall_root'])/'egs/librispeech/ASR/zipformer'
    sys.path[:0]=[str(recipe),cfg['icefall_root']]
    from optim import ScaledAdam

    train,valid=read_jsonl(out/'train.jsonl'),read_jsonl(out/'valid.jsonl')
    split=json.loads((out/'split_summary.json').read_text(encoding='utf-8'))
    if len(train)+len(valid)!=5036:raise ValueError('Expected all 5036 audio in the new split')
    for key in ('conversation_id','pcm_sha256'):
        if {r[key] for r in train}&{r[key] for r in valid}:raise ValueError('Split leakage: '+key)
    identity=stable_hash(dict(split=split['split_fingerprint'],
        settings={k:v for k,v in cfg.items() if k not in ('max_hours','resume')},
        worker=sha_file(__file__),data_worker=sha_file(Path(__file__).with_name('hynt_finetune_data.py'))))
    run_identity=out/'run_identity.json'
    if cfg.get('start_fresh') and any((out/name).exists() for name in ('last.pt','best.pt')):
        raise FileExistsError('Fresh run requested but OUTPUT_DIR already has checkpoints. Choose a new OUTPUT_DIR; previous checkpoints were not overwritten.')
    resume=cfg.get('resume') or (str(out/'last.pt') if (out/'last.pt').exists() else None)
    prior_identity=json.loads(run_identity.read_text()) if run_identity.exists() else None
    if prior_identity and prior_identity['identity']!=identity and not (resume and cfg.get('allow_code_fix_resume')):
        raise ValueError('Config/source/split changed; use a new OUTPUT_DIR instead of mixing runs')
    if prior_identity and prior_identity['identity']==identity and not cfg.get('resume') and (out/'state.json').exists():
        if json.loads((out/'state.json').read_text())['state']=='completed':
            print('This run is completed; use a new output directory or an explicit resume path.',flush=True)
            return
    device=torch.device('cuda:0')
    save_json(out/'runtime.json',dict(python=sys.version,gpu=torch.cuda.get_device_name(0),
        gpu_count=torch.cuda.device_count(),used_gpu=0,torch=torch.__version__,cuda=torch.version.cuda,
        versions={d.metadata['Name']:d.version for d in importlib.metadata.distributions()},
        icefall_commit=cfg['icefall_commit'],model_revision=REVISION,
        baseline='epoch-12-avg-8 source weights; new digit logits masked for baseline only',
        decoding='RNN-T greedy, max_symbols_per_frame=1, no LM/hotwords',fp16=True))
    print('GPU:',torch.cuda.get_device_name(0),'| train/valid:',len(train),len(valid),flush=True)
    bpe=Path(hf_hub_download(MODEL_ID,'data/Vietnam_bpe_2000_new/bpe.model',revision=REVISION,token=False))
    pretrained=Path(hf_hub_download(MODEL_ID,'exp/epoch-12-avg-8.pt',revision=REVISION,token=False))
    for path,expected in [(bpe,BPE_SHA),(pretrained,CHECKPOINT_SHA)]:
        if sha_file(path)!=expected:raise ValueError('Publisher checkpoint/tokenizer SHA256 mismatch')
    sp=spm.SentencePieceProcessor(model_file=str(bpe))
    (out/'bpe.model').write_bytes(bpe.read_bytes())
    # Only the standard pathlib class needed by this publisher's dict checkpoint.
    # Never fall back to arbitrary/unrestricted pickle loading.
    with torch.serialization.safe_globals([PosixPath]):
        checkpoint=torch.load(pretrained,map_location='cpu',weights_only=True)
    params=sherpa_params()
    # Mature schedule avoids reactivating encoder warmup; not a publisher counter.
    source_batch_count=20000
    weights=checkpoint['model'];del checkpoint;gc.collect()
    model=construct_model(params)
    model.load_state_dict(weights,strict=True)
    old_vocab=params['vocab_size']
    words=['KHÔNG','MỘT','HAI','BA','BỐN','NĂM','SÁU','BẢY','TÁM','CHÍN']
    digit_source_ids=[sp.piece_to_id('▁'+word) for word in words]
    if sp.unk_id() in digit_source_ids:raise ValueError('Number-word initialization token missing')
    added=extend_digit_bpe(bpe,out/'bpe.model')
    if added!=list('0123456789'):raise ValueError('Unexpected source digit inventory')
    sp=spm.SentencePieceProcessor(model_file=str(out/'bpe.model'))
    params['vocab_size']=sp.vocab_size()
    model=construct_model(params)
    model.load_state_dict(expand_digit_weights(torch,weights,model.state_dict(),
        digit_source_ids=digit_source_ids),strict=True)
    save_json(out/'vocabulary_adaptation.json',dict(source_vocabulary=old_vocab,
        trained_vocabulary=params['vocab_size'],added_characters=added,
        original_weights_preserved=True,source_bpe_sha256=BPE_SHA,
        digit_initialization_source_ids=digit_source_ids,new_joiner_bias_offset=-2.,
        adapted_bpe_sha256=sha_file(out/'bpe.model'),baseline_masks_new_logits=True))
    del weights;gc.collect()
    count=sum(p.numel() for p in model.parameters())
    if count>=2_000_000_000:raise ValueError('ASR parameters must be below 2B')
    model.unk_id=sp.unk_id();model.to(device)
    def ctc_fallback_report(event):
        event={**getattr(model,'ctc_context',{}),**event}
        with (out/'ctc_backend_recovery.jsonl').open('a',encoding='utf-8') as stream:
            stream.write(json.dumps(event)+'\n')
        print('CTC BACKEND RECOVERY:',event,flush=True)
    model.ctc_fallback_callback=ctc_fallback_report
    save_json(out/'model_config.json',dict(params=params,parameters=count,strict_load=True,
        source_checkpoint_sha256=CHECKPOINT_SHA,bpe_sha256=BPE_SHA,source_batch_count=source_batch_count,
        objective='0.5 simple RNN-T + pruned RNN-T; no CTC/CR head',vocabulary_adapted=True))
    print('Strict checkpoint load OK | parameters:',count,flush=True)

    # Never silently feed <unk>, blank or EOS as reference targets.
    target_ids={}
    unsupported=[]
    for row in train+valid:
        ids=sp.encode(row['training_text'],out_type=int)
        if not ids or set(ids)&{sp.unk_id(),params['blank_id'],sp.bos_id(),sp.eos_id()}:
            unsupported.append(dict(uid=row['uid'],unknown_tokens=ids.count(sp.unk_id())))
        target_ids[row['uid']]=ids
    save_json(out/'tokenizer_audit.json',dict(rows=len(target_ids),unsupported_rows=unsupported,
        text_policy='NFC, lower, punctuation/symbol to space, uppercase for original BPE; digits/accents preserved'))
    if unsupported:raise ValueError('Unsupported training labels; see tokenizer_audit.json; rows not dropped')

    fbank=Fbank(FbankConfig(num_filters=80,dither=0.0,device='cpu'))
    save_json(out/'fbank_config.json',fbank.to_dict())
    english=[r for r in train if r['english_term_hint']]
    def features(row,training=False,epoch=0,position=0):
        cache=work/'valid_features'/(row['uid']+'.npy')
        if not training and cache.exists():return torch.from_numpy(np.load(cache))
        audio,sr=sf.read(row['audio_path'],dtype='float32')
        if sr!=16000 or audio.ndim!=1:raise ValueError('Unexpected audio format')
        text=row['training_text']
        rng=np.random.default_rng(int(stable_hash([cfg['seed'],epoch,position,row['uid']])[:16],16))
        if training and cfg.get('augment_train',True):
            # Train-only concatenation of complete aligned utterances. This is
            # inspired by CS mixing, not a reproduction of phrase-level mixing.
            if english and rng.random()<cfg['cs_concat_probability']:
                other=english[int(rng.integers(len(english)))]
                if other['uid']!=row['uid'] and row['duration_s']+other['duration_s']+.1<=24:
                    second,sr2=sf.read(other['audio_path'],dtype='float32')
                    if sr2!=16000 or second.ndim!=1:raise ValueError('Invalid concatenation audio')
                    audio=np.concatenate([audio,np.zeros(1600,dtype='float32'),second])
                    text+=' '+other['training_text']
            speed=float(rng.choice([.9,1.,1.1]))
            if speed!=1.:
                audio=torchaudio.functional.resample(torch.from_numpy(audio),int(16000*speed),16000).numpy()
            if rng.random()<cfg['noise_probability']:
                rms=float(np.sqrt(np.mean(audio**2)+1e-12))
                snr=float(rng.uniform(15,25))
                audio=audio+rng.normal(0,rms/(10**(snr/20)),audio.shape).astype('float32')
                peak=float(np.max(np.abs(audio)))
                if peak>1:audio=audio/peak
        feat=fbank.extract(audio,16000).astype('float32')
        if not training:
            cache.parent.mkdir(parents=True,exist_ok=True);np.save(cache,feat)
            return torch.from_numpy(feat)
        return torch.from_numpy(feat),sp.encode(text,out_type=int)

    def greedy(encoder_out,baseline=False):
        # Initial history and one-symbol-per-frame policy follow Icefall's
        # pruned_transducer_stateless2/beam_search.py (Apache-2.0).
        ctx=params['context_size'];blank=params['blank_id']
        history=[blank]*ctx;result=[]
        decoder_input=torch.tensor([[-1]*(ctx-1)+[blank]],device=device,dtype=torch.int64)
        dec=model.joiner.decoder_proj(model.decoder(decoder_input,need_pad=False))
        enc=model.joiner.encoder_proj(encoder_out)
        for t in range(enc.shape[1]):
            logits=model.joiner(enc[:,t:t+1,:].unsqueeze(2),dec.unsqueeze(1),project_input=False)
            if baseline:logits[...,2000:]=float('-inf')
            token=int(logits.argmax().item())
            if token not in (blank,sp.unk_id()):
                result.append(token);history.append(token)
                decoder_input=torch.tensor([history[-ctx:]],device=device,dtype=torch.int64)
                dec=model.joiner.decoder_proj(model.decoder(decoder_input,need_pad=False))
        return result

    def evaluate(name,rows=None):
        rows=valid if rows is None else rows
        model.eval();set_batch_count(model,source_batch_count+global_step)
        totals=dict(n=0,n_ok=0,n_error=0,word_edits=0,n_words=0,char_edits=0,n_chars=0,exact_normalized=0,substitutions=0,deletions=0,insertions=0)
        buckets={};predictions=out/(name+'_predictions.jsonl')
        with predictions.open('w',encoding='utf-8') as stream,torch.inference_mode():
            for i,row in enumerate(rows):
                feat=features(row).unsqueeze(0).to(device)
                lengths=torch.tensor([feat.shape[1]],device=device,dtype=torch.int64)
                with torch.autocast('cuda',dtype=torch.float16):
                    enc,lens=model.forward_encoder(feat,lengths)
                    hypothesis=sp.decode(greedy(enc[:,:int(lens[0]),:],baseline=name=='baseline'))
                ref,hyp=normalize(row['reference']),normalize(hypothesis)
                ops=Levenshtein.editops(ref.split(),hyp.split())
                values=dict(n=1,n_ok=1,n_error=0,word_edits=Levenshtein.distance(ref.split(),hyp.split()),
                    n_words=len(ref.split()),char_edits=Levenshtein.distance(ref.replace(' ',''),hyp.replace(' ','')),
                    n_chars=len(ref.replace(' ','')),exact_normalized=int(ref==hyp),
                    substitutions=sum(o.tag=='replace' for o in ops),
                    deletions=sum(o.tag=='delete' for o in ops),insertions=sum(o.tag=='insert' for o in ops))
                record=dict(uid=row['uid'],conversation_id=row['conversation_id'],status='ok',
                    reference=row['reference'],hypothesis=hypothesis,english_term_hint=row['english_term_hint'],**values)
                stream.write(json.dumps(record,ensure_ascii=False)+'\n')
                for bucket in ['all',row['kind']]+(['english_term_hint'] if row['english_term_hint'] else [])+(['contains_digits'] if any(c.isdigit() for c in row['reference']) else []):
                    b=buckets.setdefault(bucket,{k:0 for k in totals})
                    for k,v in values.items():b[k]+=v
                if (i+1)%50==0:print(name,'audio',i+1,'/',len(rows),flush=True)
        for b in buckets.values():
            b['wer']=b['word_edits']/b['n_words'];b['cer']=b['char_edits']/b['n_chars']
            b['exact_match']=b['exact_normalized']/b['n']
            b['ser']=1-b['exact_match']
            b['deletion_rate']=b['deletions']/max(1,b['n_words'])
        save_json(out/(name+'_metrics.json'),dict(buckets=buckets,epoch=current_epoch,global_step=global_step,
            selection_metric=SELECTION_METRIC,
            split_fingerprint=split['split_fingerprint'],independent_test=False,
            note='English-term bucket is a fixed lexicon hint, not gold code-switch/dialect annotation'))
        score=buckets['all']
        print(name,'EM:',f"{score['exact_normalized']}/{score['n']}",
            f"({100*score['exact_match']:.3f}%)",'WER:',round(100*score['wer'],3),
            'CER:',round(100*score['cer'],3),flush=True)
        return score

    optimizer=ScaledAdam(list(model.named_parameters()),lr=cfg['learning_rate'],clipping_scale=2.0)
    scaler=torch.amp.GradScaler('cuda',init_scale=512.)
    spec_aug=SpecAugment(time_warp_factor=0,num_frame_masks=2,features_mask_size=12,
        num_feature_masks=2,frames_mask_size=20,max_frames_mask_fraction=.08)
    def training_loss(feat, ids, use_amp=True, use_masks=True):
        feat=feat.unsqueeze(0).to(device)
        lens=torch.tensor([feat.shape[1]],device=device,dtype=torch.int64)
        y=k2.RaggedTensor([ids])
        if use_masks and cfg.get('augment_train',True):feat=spec_aug(feat)
        with torch.autocast('cuda',dtype=torch.float16,enabled=use_amp):
            simple,pruned,ctc,_,cr=model(feat,lens,y,prune_range=5,am_scale=0.,lm_scale=.25,
                use_cr_ctc=False,use_spec_aug=False,spec_augment=spec_aug,
                supervision_segments=torch.tensor([[0,0,feat.shape[1]]],dtype=torch.int32),time_warp_factor=0)
            loss=(.5*simple+pruned)/max(1,len(ids))
        if not torch.isfinite(loss):
            def value(x):
                v=float(x.detach());return v if math.isfinite(v) else str(v)
            raise NonfiniteTrainingLoss(dict(amp=use_amp,specaugment=use_masks,frames=int(feat.shape[1]),
                tokens=len(ids),simple=value(simple),pruned=value(pruned),ctc='not_used',cr='not_used'))
        return loss
    def row_loss(row,epoch,position,force_fp32=False):
        model.ctc_context=dict(uid=row['uid'],epoch=epoch,position=position,global_step=global_step)
        feat,ids=features(row,True,epoch,position)
        def report(details):
            event=dict(uid=row['uid'],epoch=epoch,position=position,global_step=global_step,**details)
            with (out/'loss_recovery.jsonl').open('a',encoding='utf-8') as stream:
                stream.write(json.dumps(event,ensure_ascii=False)+'\n')
            print('LOSS RETRY:',json.dumps(event,ensure_ascii=False),flush=True)
        attempts=[]
        if not force_fp32:attempts.append(('fp16_augmented',lambda:training_loss(feat,ids)))
        attempts.extend([('fp32_augmented',lambda:training_loss(feat,ids,False)),
            ('fp32_original',lambda:training_loss(features(row),target_ids[row['uid']],False,False))])
        loss,strategy=recover_loss(attempts,report)
        if strategy!='fp16_augmented':
            print('LOSS RECOVERED:',row['uid'],strategy,flush=True)
        return loss
    global_step,current_epoch,cursor,best_wer,bad_epochs=0,0,0,None,0
    best_em=None
    if resume:
        saved=torch.load(resume,map_location='cpu',weights_only=True)
        require_em_checkpoint(saved)
        accepted_identity=identity
        if saved['identity']!=identity:
            if not cfg.get('allow_code_fix_resume'):raise ValueError('Code-fix resume is disabled')
            if saved.get('params')!=params:raise ValueError('Previous checkpoint architecture differs')
            prior=compatible_resume_metadata(resume,out,saved['identity'],cfg,split['split_fingerprint'])
            accepted_identity=saved['identity']
            save_json(out/f'previous_run_{accepted_identity}.json',prior)
            save_json(out/'code_fix_resume.json',dict(previous_identity=accepted_identity,new_identity=identity,
                checkpoint=str(Path(resume).resolve()),checkpoint_sha256=sha_file(resume),
                reason='Numerical recovery and early runtime checks; objective/split/core hyperparameters preserved',
                original_epoch=saved['epoch'],original_cursor=saved['cursor'],original_updates=saved['global_step']))
        progress=restore_training_snapshot(torch,saved,model,optimizer,scaler,accepted_identity)
        global_step,current_epoch,cursor=progress['global_step'],progress['epoch'],progress['cursor']
        best_wer,bad_epochs=progress['best_wer'],progress['bad_epochs']
        best_em=progress['best_em']
        del saved;gc.collect()
        print('Resumed epoch',current_epoch,'cursor',cursor,'updates',global_step,flush=True)
        print('Checkpoint verified: preserving best.pt; baseline is not rerun.',flush=True)
    save_json(run_identity,dict(identity=identity,config=cfg,split_fingerprint=split['split_fingerprint']))

    def save_last(epoch,next_cursor):
        # Called only after an optimizer boundary: no partial accumulated grads.
        check_finite_weights(torch,model)
        atomic_torch_save(torch,training_snapshot(torch,model,optimizer,scaler,identity,params,
            epoch=epoch,cursor=next_cursor,global_step=global_step,best_wer=best_wer,bad_epochs=bad_epochs,
            best_em=best_em,selection_metric=SELECTION_METRIC),out/'last.pt')
        save_json(out/'state.json',dict(state='checkpointed',epoch=epoch,cursor=next_cursor,
            updates=global_step,best_wer=best_wer,best_em=best_em,selection_metric=SELECTION_METRIC))

    def save_best(score,epoch,source):
        check_finite_weights(torch,model)
        atomic_torch_save(torch,dict(model=model.state_dict(),params=params,identity=identity,
            wer=score['wer'],exact_match=score['exact_match'],exact_normalized=score['exact_normalized'],
            n=score['n'],selection_metric=SELECTION_METRIC,epoch=epoch,
            global_step=global_step,source=source),out/'best.pt')

    if best_wer is None:
        # Test the largest supported augmented feature shape, actual optimizer
        # updates and the same checkpoint/restore path before full validation.
        # Restore original weights/optimizer/RNG afterwards: baseline is untouched.
        print('PREFLIGHT: long-shape loss/backward + 2 updates + checkpoint roundtrip',flush=True)
        model.ctc_context=dict(stage='preflight')
        original={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        original_rng=(torch.get_rng_state(),torch.cuda.get_rng_state_all(),random.getstate())
        model.train();set_batch_count(model,source_batch_count)
        probe_row=max(train,key=lambda r:r['duration_s'])
        feat,ids=features(probe_row,True,0,0)
        # Concat <=24 seconds then speed >=0.9 => <=26.67 seconds. Padding is a
        # memory/shape stress fixture only, never part of training/quality metrics.
        if feat.shape[0]<2700:feat=torch.nn.functional.pad(feat,(0,0,0,2700-feat.shape[0]))
        successful=0
        for attempt in range(8):
            optimizer.zero_grad(set_to_none=True)
            old_scale=scaler.get_scale()
            for micro in range(2):
                probe_loss,_=recover_loss([
                    ('fp16_stress',lambda:training_loss(feat,ids)),
                    ('fp32_stress',lambda:training_loss(feat,ids,False))],
                    lambda details:print('PREFLIGHT LOSS RETRY:',details,flush=True))
                probe_loss=probe_loss/2
                scaler.scale(probe_loss).backward()
            scaler.step(optimizer);scaler.update()
            if scaler.get_scale()>=old_scale:successful+=1
            if successful==2:break
        if successful!=2:raise FloatingPointError('CUDA preflight could not complete 2 finite optimizer updates')
        check_finite_weights(torch,model)
        changed=any(not torch.equal(p.detach().cpu(),original[name]) for name,p in model.named_parameters())
        if not changed:raise RuntimeError('Preflight optimizer did not change weights')
        optimizer.zero_grad(set_to_none=True)
        probe_loss_value=float(probe_loss.detach());del probe_loss
        probe_path=out/'_preflight_roundtrip.pt'
        atomic_torch_save(torch,training_snapshot(torch,model,optimizer,scaler,identity,params,
            epoch=1,cursor=0,global_step=successful,best_wer=None,bad_epochs=0),probe_path)
        restored=torch.load(probe_path,map_location='cpu',weights_only=True)
        progress=restore_training_snapshot(torch,restored,model,optimizer,scaler,identity)
        if progress['global_step']!=successful:raise ValueError('Checkpoint update counter mismatch')
        model.load_state_dict(original,strict=True)
        optimizer=ScaledAdam(list(model.named_parameters()),lr=cfg['learning_rate'],clipping_scale=2.0)
        scaler=torch.amp.GradScaler('cuda',init_scale=512.)
        torch.set_rng_state(original_rng[0]);torch.cuda.set_rng_state_all(original_rng[1]);random.setstate(original_rng[2])
        save_json(out/'training_probe.json',dict(forward_backward=True,optimizer_updates=successful,
            weights_changed=changed,checkpoint_save_load=True,baseline_weights_restored=True,
            accumulated_micro_batches=2,feature_frames=int(feat.shape[0]),loss=probe_loss_value,gpu=torch.cuda.get_device_name(0),
            peak_reserved_bytes=torch.cuda.max_memory_reserved()))
        del restored,original,feat;probe_path.unlink();gc.collect();torch.cuda.empty_cache()
        print('PREFLIGHT OK; original hynt weights restored. Starting baseline validation.',flush=True)
        score=evaluate('baseline')
        best_wer,best_em=score['wer'],score['exact_match']
        save_best(score,0,'baseline')
        save_last(1,0)
        current_epoch=1
    hard_path=out/'hard_pool.json'
    hard_uids=[]
    if hard_path.exists():
        pool=json.loads(hard_path.read_text(encoding='utf-8'))
        if pool['identity']!=identity:raise ValueError('Hard pool belongs to another run')
        hard_uids=pool['uids']
    if current_epoch>1 and cfg.get('hard_extra_fraction',.2)>0 and not hard_path.exists():
        raise ValueError('Keep hard_pool.json with last.pt to resume this method')
    started=time.monotonic();accum=cfg['gradient_accumulation']
    # OOM/ABI/model mismatch errors propagate with the current checkpoint intact.
    for epoch in range(max(1,current_epoch),cfg['epochs']+1):
        current_epoch=epoch
        order=replay_order(train,cfg['seed'],epoch,hard_uids,cfg['english_extra_fraction'],cfg.get('hard_extra_fraction',.2))
        start=cursor
        optimizer.zero_grad(set_to_none=True);model.train()
        planned=len(train)+round(len(english)*cfg['english_extra_fraction'])+round(len(train)*cfg.get('hard_extra_fraction',.2))
        updates_per_epoch=math.ceil(planned/accum)
        for offset in range(start,len(order),accum):
            batch_indices=order[offset:offset+accum]
            lr=cfg['learning_rate']*(.5+.5*math.cos(math.pi*min(1.,global_step/(cfg['epochs']*updates_per_epoch))))
            for group in optimizer.param_groups:group['lr']=max(lr,cfg['learning_rate']*.1)
            for attempt in range(8):
                loss_value=0.
                for micro,index in enumerate(batch_indices):
                    row=train[index]
                    set_batch_count(model,source_batch_count+global_step)
                    loss=row_loss(row,epoch,offset+micro,force_fp32=attempt>0)/len(batch_indices)
                    scaler.scale(loss).backward();loss_value+=float(loss.detach())
                old_scale=scaler.get_scale()
                scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
                if scaler.get_scale()>=old_scale:
                    global_step+=1;break
                print('AMP overflow: retry same accumulation group at scale',scaler.get_scale(),flush=True)
            else:raise FloatingPointError('8 failed AMP attempts; group was not advanced/skipped')
            cursor=offset+len(batch_indices)
            if global_step%10==0:
                print(f'epoch {epoch}/{cfg["epochs"]} | audio {cursor}/{len(order)} | update {global_step} | loss {loss_value:.4f} | lr {lr:.6g}',flush=True)
            if global_step and global_step%cfg['save_every_updates']==0:save_last(epoch,cursor)
            if time.monotonic()-started>cfg['max_hours']*3600:
                save_last(epoch,cursor)
                # Protect the resumable checkpoint before validation. A partial
                # epoch still needs a measured result for the report deadline.
                name=f'partial_epoch_{epoch:02}_step_{global_step:06}'
                print('Train time budget reached; checkpoint saved. Validating:',name,flush=True)
                score=evaluate(name)
                if better_validation_score(score['exact_match'],score['wer'],best_em,best_wer):
                    best_wer,best_em=score['wer'],score['exact_match'];bad_epochs=0
                    save_best(score,epoch,'fine_tuned_partial_epoch')
                save_last(epoch,cursor)
                save_json(out/'state.json',dict(state='paused_time_budget',epoch=epoch,cursor=cursor,
                    updates=global_step,best_wer=best_wer,best_em=best_em,selection_metric=SELECTION_METRIC,
                    validation_run=name,partial_epoch=True,
                    independent_test=False,peak_vram_bytes=torch.cuda.max_memory_reserved()))
                return
        score=evaluate(f'epoch_{epoch:02}')
        if global_step==0:raise RuntimeError('No successful optimizer update; inspect training precision/scaler')
        if better_validation_score(score['exact_match'],score['wer'],best_em,best_wer):
            best_wer,best_em=score['wer'],score['exact_match'];bad_epochs=0
            save_best(score,epoch,'fine_tuned')
        else:bad_epochs+=1
        if epoch==1 and cfg.get('hard_extra_fraction',.2)>0:
            print('Building error pool using TRAIN only; not dialect labels.',flush=True)
            evaluate('train_mining_epoch_01',train)
            records=read_jsonl(out/'train_mining_epoch_01_predictions.jsonl')
            errors=[r for r in records if not r['exact_normalized']]
            errors.sort(key=lambda r:(-r['deletions']/max(1,r['n_words']),
                -r['word_edits']/max(1,r['n_words']),r['uid']))
            hard_uids=[r['uid'] for r in errors[:math.ceil(len(train)*.3)]]
            save_json(hard_path,dict(identity=identity,uids=hard_uids,source='train_mining_epoch_01',
                note='JTT-inspired continuous fine-tuning; not a reproduction, not inferred dialect labels'))
        cursor=0;save_last(epoch+1,0)
        if bad_epochs>=cfg['patience']:
            print('Early stopping by EM, then WER; best.pt may still be the baseline.',flush=True)
            break
    save_json(out/'state.json',dict(state='completed',updates=global_step,best_wer=best_wer,
        best_em=best_em,selection_metric=SELECTION_METRIC,
        peak_vram_bytes=torch.cuda.max_memory_reserved(),independent_test=False))


if __name__=='__main__':
    p=argparse.ArgumentParser(description=__doc__);p.add_argument('config')
    args=p.parse_args()
    try:main(args.config)
    except Exception:
        import traceback
        cfg=json.loads(Path(args.config).read_text(encoding='utf-8'))
        save_json(Path(cfg['output'])/'failure.json',dict(state='failed',traceback=traceback.format_exc()))
        raise
''',
    'hynt_finetune_data.py': r'''"""Prepare a new train-only ASR experiment, including the historical 991 rows.

Does not change the team's manifests or read public-test/environment records.
Conversation and duplicate PCM components stay in one partition. No speaker
identity is inferred from folders, transcripts, or acoustic similarity.
"""
import argparse
from collections import defaultdict
import hashlib
import io
import json
from pathlib import Path, PurePosixPath
import unicodedata
import wave
import zipfile

ENGLISH_TERMS = ('youtube','facebook','zalo','gmail','google','wifi','bluetooth',
                 'spotify','netflix','tiktok','messenger','zoom','app','email',
                 'online','offline','chrome','excel','word','powerpoint')


def normalize(text):
    text = unicodedata.normalize('NFC', str(text)).lower()
    return ' '.join(''.join(' ' if unicodedata.category(c)[0] in 'PS' else c for c in text).split())


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     separators=(',', ':')).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
    temp.replace(path)


def english_hint(text):
    return bool(set(normalize(text).split()) & set(ENGLISH_TERMS))


class Source:
    def __init__(self, path):
        path = Path(path).resolve()
        self.archive = None
        if path.is_file() and path.suffix.lower()=='.zip':
            self.archive = zipfile.ZipFile(path)
            names = self.archive.namelist()
            if len(names)!=len(set(names)): raise ValueError('Duplicate ZIP members')
            candidates = [n for n in names if PurePosixPath(n).parts[-2:]==('train','train.jsonl')
                          and '__MACOSX' not in PurePosixPath(n).parts]
            if len(candidates)!=1: raise ValueError('Need one train/train.jsonl')
            self.manifest = candidates[0]
            self.train = PurePosixPath(self.manifest).parent
        else:
            candidates = [path/'train/train.jsonl', path/'train+public_test/train/train.jsonl',
                          path/'train.jsonl']
            candidates = [p for p in candidates if p.is_file() and p.parent.name=='train']
            if len(candidates)!=1: raise FileNotFoundError('Expected train/train.jsonl in DATASET_ROOT')
            self.manifest = candidates[0]
            self.train = self.manifest.parent
        self.path = path

    def read_manifest(self):
        return (self.archive.read(self.manifest) if self.archive else self.manifest.read_bytes())

    def audio(self, relative):
        p = PurePosixPath(relative)
        if (p.is_absolute() or '..' in p.parts or '\\' in relative or ':' in relative
                or not p.parts or p.parts[0]!='audio' or p.suffix.lower()!='.wav'):
            raise ValueError('Audio must stay in train/audio')
        if self.archive: return self.archive.read(str(self.train/p))
        target = (self.train/Path(*p.parts)).resolve()
        if not target.is_relative_to(self.train.resolve()): raise ValueError('Audio escapes train')
        return target.read_bytes()

    def close(self):
        if self.archive: self.archive.close()


def split_rows(rows, valid_fraction=.1, seed=2026):
    if not 0 < valid_fraction < 1: raise ValueError('valid_fraction must be between 0 and 1')
    parent = {r['conversation_id']: r['conversation_id'] for r in rows}
    def find(x):
        while parent[x]!=x:
            parent[x] = parent[parent[x]]; x=parent[x]
        return x
    seen = {}
    for r in rows:
        if r['pcm_sha256'] in seen:
            a,b=find(r['conversation_id']), find(seen[r['pcm_sha256']])
            parent[max(a,b)] = min(a,b)
        else: seen[r['pcm_sha256']]=r['conversation_id']
    groups = defaultdict(list)
    for r in rows: groups[find(r['conversation_id'])].append(r)
    if len(groups)<2: raise ValueError('Need two independent conversation/PCM groups')
    keys = sorted(groups, key=lambda x: stable_hash([seed,x]))
    target = max(1, round(len(rows)*valid_fraction))
    chosen, count = set(), 0
    for k in keys:
        if count+len(groups[k])<=target and len(chosen)<len(groups)-1:
            chosen.add(k); count+=len(groups[k])
    if not chosen: chosen={min(keys,key=lambda k:abs(len(groups[k])-target))}
    train,valid=[],[]
    for k in keys:
        (valid if k in chosen else train).extend(groups[k])
    if not train or not valid: raise ValueError('Empty partition')
    for key in ('conversation_id','pcm_sha256','audio_relative'):
        if {r[key] for r in train} & {r[key] for r in valid}:
            raise ValueError('Cross-partition overlap: '+key)
    return sorted(train,key=lambda r:r['uid']), sorted(valid,key=lambda r:r['uid'])


def prepare(dataset, work, output, valid_fraction=.1, seed=2026, expected_audio=5036):
    work, output = Path(work).resolve(), Path(output).resolve()
    if work==output: raise ValueError('Use separate work and result directories')
    source = Source(dataset)
    rows, seen, ids = [], {}, set()
    content = source.read_manifest()
    try:
        for line_no,line in enumerate(content.decode('utf-8-sig').splitlines(),1):
            if not line.strip(): continue
            record=json.loads(line)
            sid=record['id']
            if not isinstance(sid,str) or sid in ids: raise ValueError('Invalid/duplicate conversation ID')
            ids.add(sid)
            turns=[('final',None,record)]+[('history',i,t) for i,t in enumerate(record.get('history',[]))
                   if t.get('role') in ('user','human') and t.get('audio')]
            for kind,index,turn in turns:
                relative=turn.get('audio'); text=turn.get('audio_text',turn.get('text'))
                if not isinstance(relative,str) or not isinstance(text,str) or not normalize(text):
                    raise ValueError('Missing audio/transcript; repair source, do not drop rows')
                if relative in seen:
                    raise ValueError('Repeated manifest audio path; resolve ownership before splitting')
                payload=source.audio(relative)
                with wave.open(io.BytesIO(payload),'rb') as w:
                    sr,frames=w.getframerate(),w.getnframes()
                    if sr!=16000 or w.getnchannels()!=1 or w.getsampwidth()!=2 or w.getcomptype()!='NONE':
                        raise ValueError('Requires mono 16kHz PCM16 WAV')
                    pcm=w.readframes(frames)
                    if len(pcm)!=frames*2 or frames<=0: raise ValueError('Invalid WAV payload')
                target=work/Path(*PurePosixPath(relative).parts)
                if not target.is_relative_to(work): raise ValueError('Output audio escapes work')
                target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists() or sha_file(target)!=hashlib.sha256(payload).hexdigest():
                    target.write_bytes(payload)
                row=dict(uid=stable_hash([sid,kind,index,relative])[:24], conversation_id=sid,
                    manifest_line=line_no, kind=kind, history_index=index,
                    audio_relative=relative, audio_path=str(target), duration_s=frames/sr,
                    pcm_sha256=hashlib.sha256(pcm).hexdigest(),
                    wav_sha256=hashlib.sha256(payload).hexdigest(), reference=text,
                    training_text=normalize(text).upper(), english_term_hint=english_hint(text))
                seen[relative]=row; rows.append(row)
    finally: source.close()
    if len(rows)!=expected_audio:
        raise ValueError(f'Expected {expected_audio} train audio, got {len(rows)}; no silent subset')
    train,valid=split_rows(rows,valid_fraction,seed)
    identity=[{k:r[k] for k in ('uid','conversation_id','audio_relative','pcm_sha256','reference')}
              for r in sorted(rows,key=lambda r:r['uid'])]
    fingerprint=stable_hash(dict(rows=identity,seed=seed,valid_fraction=valid_fraction,
        train_ids=[r['uid'] for r in train],valid_ids=[r['uid'] for r in valid]))
    output.mkdir(parents=True,exist_ok=True)
    old=output/'split_summary.json'
    if old.exists() and json.loads(old.read_text(encoding='utf-8'))['split_fingerprint']!=fingerprint:
        raise ValueError('Existing split differs; use a new output directory')
    for name,part in [('train',train),('valid',valid)]:
        (output/(name+'.jsonl')).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in part),encoding='utf-8')
    summary=dict(total_audio=len(rows),total_conversations=len(ids),train_audio=len(train),valid_audio=len(valid),
        train_conversations=len({r['conversation_id'] for r in train}),valid_conversations=len({r['conversation_id'] for r in valid}),
        train_hours=sum(r['duration_s'] for r in train)/3600,valid_hours=sum(r['duration_s'] for r in valid)/3600,
        unique_pcm=len({r['pcm_sha256'] for r in rows}),seed=seed,valid_fraction_requested=valid_fraction,
        split_fingerprint=fingerprint,manifest_sha256=hashlib.sha256(content).hexdigest(),
        train_english_term_hints=sum(r['english_term_hint'] for r in train),
        valid_english_term_hints=sum(r['english_term_hint'] for r in valid),
        conversation_disjoint=True,pcm_disjoint=True,speaker_disjoint_verified=False,
        includes_historical_991=True, independent_test=False,
        protocol='New development train/valid split of all 5036 train WAVs; historical 991 reused by user request',
        source=str(Path(dataset).resolve()))
    save_json(old,summary)
    print(json.dumps({k:v for k,v in summary.items() if k!='source'},ensure_ascii=False,indent=2),flush=True)
    return summary


if __name__=='__main__':
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--dataset',required=True); p.add_argument('--work',required=True)
    p.add_argument('--output',required=True);p.add_argument('--valid-fraction',type=float,default=.1)
    p.add_argument('--seed',type=int,default=2026)
    a=p.parse_args();prepare(a.dataset,a.work,a.output,a.valid_fraction,a.seed)
''',
}

CODE_DIR=WORK_DIR/'code'
CODE_DIR.mkdir(exist_ok=True)
for name,source in SOURCES.items():
    (CODE_DIR/name).write_text(source,encoding='utf-8')
    old=OUTPUT_DIR/name
    if old.is_file() and old.read_text(encoding='utf-8')!=source:
        import hashlib
        digest=hashlib.sha256(old.read_bytes()).hexdigest()[:16]
        backup=OUTPUT_DIR/'previous_code'/digest
        backup.mkdir(parents=True,exist_ok=True)
        (backup/name).write_bytes(old.read_bytes())
    (OUTPUT_DIR/name).write_text(source,encoding='utf-8')

## Cell 6 — 4532 train / 504 valid; cùng seed và fingerprint với hynt.


In [ ]:
candidates=[]
if DATASET_ROOT.exists():
    if DATASET_ROOT.is_file():candidates=[DATASET_ROOT]
    elif any(p.is_file() for p in [DATASET_ROOT/'train/train.jsonl',DATASET_ROOT/'train+public_test/train/train.jsonl']):
        candidates=[DATASET_ROOT]
    else:candidates=list(DATASET_ROOT.glob('train+public_test.zip'))
if not candidates:
    candidates=[p.parent.parent for p in DATASET_ROOT.rglob('train.jsonl')
                if p.parent.name=='train' and '__MACOSX' not in p.parts
                and not any(x.endswith('_environment') for x in p.parts)]
if not candidates:candidates=list(DATASET_ROOT.rglob('train+public_test.zip'))
candidates=list(dict.fromkeys(candidates))
if len(candidates)!=1:raise FileNotFoundError('Add the LifeMate dataset Input or set DATASET_ROOT to its unique train root/ZIP')
SOURCE_DATASET=candidates[0]
run([PYTHON,CODE_DIR/'hynt_finetune_data.py','--dataset',SOURCE_DATASET,
     '--work',WORK_DIR/'data','--output',OUTPUT_DIR,
     '--valid-fraction',str(VALID_FRACTION),'--seed',str(SEED)])
split=json.loads((OUTPUT_DIR/'split_summary.json').read_text(encoding='utf-8'))
print('Audio train/valid:',split['train_audio'],split['valid_audio'])
print('Overlap conversation/PCM: 0; speaker-disjoint chưa xác minh')

## Cell 7 — Ghi config. Không dùng checkpoint hynt hoặc run WER-selected.


In [ ]:
if RESUME_DIR is None and AUTO_RESUME_DISCOVERY:
    if (OUTPUT_DIR/'last.pt').is_file() and (OUTPUT_DIR/'best.pt').is_file():
        RESUME_DIR=OUTPUT_DIR
    else:
        matches=[]
        for candidate in Path('/kaggle/input').rglob('last.pt'):
            folder=candidate.parent
            if not all((folder/name).is_file() for name in ('best.pt','run_identity.json','split_summary.json')):
                continue
            previous_split=json.loads((folder/'split_summary.json').read_text(encoding='utf-8'))
            prior=json.loads((folder/'run_identity.json').read_text(encoding='utf-8'))
            if (previous_split.get('split_fingerprint')==split['split_fingerprint']
                    and prior.get('config',{}).get('selection_metric')==SELECTION_METRIC):
                matches.append(folder)
        matches=list(dict.fromkeys(matches))
        if len(matches)>1:raise ValueError('Multiple matching runs: set RESUME_DIR to the one you want: '+str(matches))
        if matches:RESUME_DIR=matches[0]
if RESUME_DIR is not None:
    RESUME_DIR=Path(RESUME_DIR)
    if not (RESUME_DIR/'last.pt').is_file() or not (RESUME_DIR/'best.pt').is_file():
        raise FileNotFoundError('RESUME_DIR needs both last.pt and best.pt from the same run')
    if not (RESUME_DIR/'run_identity.json').is_file():
        raise FileNotFoundError('Keep run_identity.json with last.pt to verify old-run settings')
    previous=json.loads((RESUME_DIR/'run_identity.json').read_text(encoding='utf-8'))
    expected_core=dict(selection_metric=SELECTION_METRIC,hard_extra_fraction=HARD_EXTRA_FRACTION,augment_train=AUGMENT_TRAIN,icefall_commit=ICEFALL_COMMIT,seed=SEED,epochs=EPOCHS,
        gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,
        patience=PATIENCE,english_extra_fraction=ENGLISH_EXTRA_FRACTION,
        cs_concat_probability=CS_CONCAT_PROBABILITY,noise_probability=NOISE_PROBABILITY)
    if previous.get('split_fingerprint')!=split['split_fingerprint'] or any(
            previous.get('config',{}).get(key)!=value for key,value in expected_core.items()):
        raise ValueError('Resume split/settings differ; best.pt was not copied or overwritten')
    if RESUME_DIR.resolve()!=OUTPUT_DIR.resolve() and (OUTPUT_DIR/'best.pt').exists():
        raise FileExistsError('OUTPUT_DIR already has best.pt; refusing to overwrite it with another run')
    print('RESUME:',RESUME_DIR,'| keep best.pt and continue last.pt; baseline will not be rerun')
    for path in list(RESUME_DIR.glob('*_metrics.json'))+[RESUME_DIR/'best.pt',RESUME_DIR/'baseline_predictions.jsonl']:
        if path.is_file() and path.resolve()!=(OUTPUT_DIR/path.name).resolve():
            shutil.copy2(path,OUTPUT_DIR/path.name)
config=dict(work=str(WORK_DIR),output=str(OUTPUT_DIR),icefall_root=str(ICEFALL_ROOT),
    selection_metric=SELECTION_METRIC,hard_extra_fraction=HARD_EXTRA_FRACTION,augment_train=AUGMENT_TRAIN,
    icefall_commit=ICEFALL_COMMIT,seed=SEED,epochs=EPOCHS,
    gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,
    patience=PATIENCE,max_hours=MAX_HOURS,save_every_updates=SAVE_EVERY_UPDATES,
    english_extra_fraction=ENGLISH_EXTRA_FRACTION,cs_concat_probability=CS_CONCAT_PROBABILITY,
    noise_probability=NOISE_PROBABILITY,resume=str(RESUME_DIR/'last.pt') if RESUME_DIR else None,
    allow_code_fix_resume=ALLOW_CODE_FIX_RESUME,
    start_fresh=not AUTO_RESUME_DISCOVERY and RESUME_DIR is None)
CONFIG_PATH=OUTPUT_DIR/'train_config.json'
CONFIG_PATH.write_text(json.dumps(config,indent=2),encoding='utf-8')

## Cell 8 — Preflight → baseline → epoch 1 → mining train → replay epoch 2–5. Không dùng valid để mining.


In [ ]:
try:
    run([PYTHON,'-u',CODE_DIR/'sherpa_vi_finetune.py',CONFIG_PATH])
finally:
    for name in ('best.pt','last.pt','failure.json','loss_recovery.jsonl','ctc_backend_recovery.jsonl','code_fix_resume.json'):
        path=OUTPUT_DIR/name
        if path.exists():print('KEPT:',path,'| bytes:',path.stat().st_size)

## Cell 9 — Số câu đúng, EM%, WER/CER/SER. train_mining là số trên train, không phải kết quả valid.


In [ ]:
import hashlib
summary=[]
for path in sorted(OUTPUT_DIR.glob('*_metrics.json')):
    if path.name.startswith('train_mining_'):continue
    item=json.loads(path.read_text(encoding='utf-8'));row=item['buckets']['all']
    summary.append(dict(run=path.stem,n=row['n'],WER_pct=100*row['wer'],CER_pct=100*row['cer'],
                        exact=row['exact_normalized'],
                        EM_pct=100*row['exact_normalized']/row['n'],
                        SER_pct=100*(1-row['exact_normalized']/row['n']),epoch=item['epoch']))
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('State:',(OUTPUT_DIR/'state.json').read_text(encoding='utf-8'))
print('Checkpoints:',[p.name for p in OUTPUT_DIR.glob('*.pt')])
for name,source in SOURCES.items():
    (OUTPUT_DIR/name).write_text(source,encoding='utf-8')
def file_sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
metadata={p.name:file_sha256(p) for p in OUTPUT_DIR.iterdir()
          if p.is_file() and p.name!='metadata_hashes.json'
          and p.suffix in ('.json','.jsonl','.py','.pt','.model')}
(OUTPUT_DIR/'metadata_hashes.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Kết quả ở:',OUTPUT_DIR,'| Chưa có test độc lập; không so thẳng với WER 15,34% trên 991 mẫu cũ.')